# FinalTest: 联合命名实体识别 + 关系抽取

> **杂粮作物领域 · NER(12类实体) + RE(6类关系)**

| 项目 | 说明 |
|------|------|
| 模型 | DeBERTa-v3-large + CRF (NER) / Entity Markers (RE) |
| 数据 | HW_train_data.json (800条) |
| 评分 | Total = 0.4×NER_Score + 0.6×RE_Score |

## 运行说明

按顺序从上到下执行每个 Code Cell。

- **Cell 1**: 检查环境
- **Cell 2**: 安装依赖
- **Cell 3**: 查看数据分布
- **Cell 4**: 预下载模型 (~6-8 min)
- **Cell 5**: 训练 NER (~12 min, 8 epochs)
- **Cell 6**: 训练 RE (~25-35 min, 6 epochs)
- **Cell 7**: 联合推理 + 评估 (~3 min)


### Cell 1: 环境检查

**预期输出**: CUDA=True, GPU信息, 文件列表

In [ ]:
# 设置 HuggingFace 镜像源（国内平台必须）
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# 清理可能存在的坏缓存
import shutil
for d in ['/root/.cache/huggingface/hub/models--microsoft--deberta-v3-large']:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f'Cleared bad cache: {d}')

import torch, sys

print("=" * 50)
print(" Environment")
print("=" * 50)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory:  {mem:.1f} GB")
else:
    print("WARNING: CUDA is False. Do not start training until the runtime uses a GPU PyTorch build.")

print(f"Files:   {sorted(os.listdir('.'))}")
print("=" * 50)


### Cell 2: 安装依赖

安装 transformers, sentencepiece, pytorch-crf, scikit-learn, tqdm

In [ ]:
!pip install "transformers<5" sentencepiece protobuf pytorch-crf scikit-learn tqdm -q
import torch, transformers
print(f"torch {torch.__version__}, transformers {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("Dependencies OK")


### Cell 3: 数据快速检查

**预期输出**: 800 样本, 实体/关系统计

In [ ]:
import json
from collections import Counter

data = json.load(open("HW_train_data.json"))
print(f"样本数: {len(data)}")

ent_counter = Counter()
for d in data:
    for e in d["entities"]:
        ent_counter[e["label"]] += 1
print(f"实体类型: {len(ent_counter)}, 总实体: {sum(ent_counter.values())}")

rel_counter = Counter()
for d in data:
    for r in d["relations"]:
        rel_counter[r["label"]] += 1
print(f"关系类型: {len(rel_counter)}, 总关系: {sum(rel_counter.values())}")
print(f"\n数据 OK")


### Cell 4: 预下载模型

把模型下载到可见文件夹，只下一次，NER和RE共用。

**预计耗时**: ~6-8 分钟

In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

from huggingface_hub import snapshot_download

print("Downloading microsoft/deberta-v3-large...")
snapshot_download(
    "microsoft/deberta-v3-large",
    local_dir="./deberta-v3-large",
    local_dir_use_symlinks=False,
    ignore_patterns=["*.msgpack", "*.h5"],
)
print("✅ Downloaded to ./deberta-v3-large")


### Cell 5: 训练 NER (8 epochs)

使用 DeBERTa-v3-large + CRF 做序列标注。

| 参数 | 值 |
|------|-----|
| model | ./deberta-v3-large |
| epochs | 8 |
| batch_size | 12 |
| lr | 1e-5 |
| patience | 2 |
| val_ratio | 0.05 |


In [ ]:
import torch
assert torch.cuda.is_available(), "CUDA=False：当前不是 GPU PyTorch 环境，请先切换/修复运行时再训练。"

!python train_ner.py \
    --data HW_train_data.json \
    --model ./deberta-v3-large \
    --save_dir ./output_ner \
    --max_length 384 \
    --batch_size 12 \
    --epochs 8 \
    --lr 1e-5 \
    --warmup_ratio 0.1 \
    --dropout 0.1 \
    --val_ratio 0.05 \
    --patience 2

import os
assert os.path.exists("./output_ner/best_ner_model.pt"), "NER model not saved!"
print(f"NER model: {os.path.getsize('./output_ner/best_ner_model.pt')/1e9:.2f} GB")
print("NER PASSED")


### Cell 6: 训练 RE (6 epochs, neg_ratio=1)

使用 DeBERTa-v3-large + Entity Markers 做关系分类。

| 参数 | 值 |
|------|-----|
| model | ./deberta-v3-large |
| epochs | 6 |
| batch_size | 12 |
| lr | 1e-5 |
| neg_ratio | 1 |
| patience | 2 |
| val_ratio | 0.05 |


In [ ]:
import torch
assert torch.cuda.is_available(), "CUDA=False：当前不是 GPU PyTorch 环境，请先切换/修复运行时再训练。"

!python train_re.py \
    --data HW_train_data.json \
    --model ./deberta-v3-large \
    --save_dir ./output_re \
    --max_length 384 \
    --batch_size 12 \
    --epochs 6 \
    --lr 1e-5 \
    --warmup_ratio 0.1 \
    --dropout 0.1 \
    --neg_ratio 1 \
    --val_ratio 0.05 \
    --patience 2

import os
assert os.path.exists("./output_re/best_re_model.pt"), "RE model not saved!"
print(f"RE model: {os.path.getsize('./output_re/best_re_model.pt')/1e9:.2f} GB")
print("RE PASSED")


### Cell 7: 联合推理 + 评估

加载 NER + RE 模型，跑完整 Pipeline。

**预期输出**: predictions.json + P/R/F1/Score


In [ ]:
!python predict.py \
    --ner_model_dir ./output_ner \
    --re_model_dir ./output_re \
    --data HW_train_data.json \
    --output predictions.json \
    --batch_size 12 \
    --re_threshold 0.9 \
    --max_length 384 \
    --val_ratio 0.05 \
    --evaluate

import os, json
assert os.path.exists("predictions.json"), "predictions.json not found!"

with open("predictions.json") as f:
    preds = json.load(f)
total_ent = sum(len(p['entities']) for p in preds)
total_rel = sum(len(p['relations']) for p in preds)
print(f"\n预测: {len(preds)} 文本, {total_ent} 实体, {total_rel} 关系")
print("Inference PASSED")


### 完成！🎉

所有训练和推理已完成。关键输出文件:

| 文件 | 说明 |
|------|------|
| `output_ner/best_ner_model.pt` | NER 最佳模型 |
| `output_re/best_re_model.pt` | RE 最佳模型 |
| `predictions.json` | 预测结果 |

**查看最终分数**: 向上翻到 Cell 7 的输出末尾，找到 `Total Score` 行。
